In [ ]:
!RNAdistance --version
#RNAdistance 2.3.0cuda

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
DATA_PATH = "../../../data/archiveII/"
dref_all = pd.read_csv(DATA_PATH +  "archiveII.csv", index_col="id")
display(dref_all.head(3))
display(dref_all.shape)

In [ ]:
dref = dref_all

Results = pd.DataFrame(np.zeros((len(dref.index),len(dref.index))))-1
Results.index = dref.index
Results.columns = dref.index
for i,rowi in tqdm(dref.iterrows(),total=len(dref)):
    for j,rowj in dref.iterrows():
        if Results.loc[j,i] < 0:
            echo_line = rowi.structure + "\n" + rowj.structure
            # RNAdistance cannot handle pseudoknots (treated as unpaired) https://pubmed.ncbi.nlm.nih.gov/36077037/
            echo_line = echo_line.replace('<','.').replace('>','.').replace('{','.').replace('}','.')
            oo = subprocess.check_output(["RNAdistance"], input=echo_line.encode('utf-8'))

            if oo != b"":
                max_len = len(rowi.structure) if len(rowi.structure)>len(rowj.structure) else len(rowj.structure) 
                Results.loc[i,j] = float(oo[2:])/max_len
            else:
                Results.loc[i,j] = np.nan
                print("Warning: RNAdistance error with {} and {}".format(i, j))
        else:
            Results.loc[i,j] = Results.loc[j,i]

In [ ]:
os.makedirs(DATA_PATH + "distances/", exist_ok=True)
Results.to_csv(DATA_PATH + "distances/rnadist_f_all.csv")

In [ ]:
Results.to_hdf(DATA_PATH + "distances/rnadist_f_all.h5", key='rnadist', mode='w')